### 載入套件和資料

In [1]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import roc_auc_score, average_precision_score, accuracy_score
import lightgbm as lgb

df = pd.read_parquet('../data/processed/model_table.parquet')

train = df[df['split'] == 'train']
test = df[df['split'] == 'test']
y_train = train['label']
y_test = test['label']

num_features = [
    'n_views_before_cart', 'n_products_viewed_before_cart', 'secs_to_first_cart',
    'cart_hour', 'cart_dow', 'cart_price', 'cart_brand_missing'
]

print(train.shape, test.shape)

(463222, 14) (109875, 14)


### 評估函式

In [2]:
def evaluate(name, y_true, y_prob):
    return {
        'model': name,
        'roc_auc': round(roc_auc_score(y_true, y_prob), 4),
        'pr_auc': round(average_precision_score(y_true, y_prob), 4),
        'accuracy': round(accuracy_score(y_true, (y_prob >= 0.5).astype(int)), 4),
    }

results = []

### Logistic regression baseline

In [3]:
lr = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
lr.fit(train[num_features], y_train)

p_lr = lr.predict_proba(test[num_features])[:, 1]
results.append(evaluate('Logistic regression', y_test, p_lr))
pd.DataFrame(results)

,model,roc_auc,pr_auc,accuracy
0,Logistic regression,0.5445,0.5497,0.5376


### LightGBM

In [4]:
cats = train['cart_category_id'].astype('category').cat.categories

def prep(d):
    X = d[num_features].copy()
    X['cart_category_id'] = pd.Categorical(d['cart_category_id'], categories=cats)
    return X

X_train, X_test = prep(train), prep(test)

model = lgb.LGBMClassifier(n_estimators=300, learning_rate=0.05, num_leaves=31, verbose=-1)
model.fit(X_train, y_train)

p_lgb = model.predict_proba(X_test)[:, 1]
results.append(evaluate('LightGBM', y_test, p_lgb))
pd.DataFrame(results)

/var/folders/dc/03c0xx517lscty57znvb7_8m0000gn/T/ipykernel_30314/2494177621.py:5: Pandas4Warning: Constructing a Categorical with a dtype and values containing non-null entries not in that dtype's categories is deprecated and will raise in a future version.
  X['cart_category_id'] = pd.Categorical(d['cart_category_id'], categories=cats)


,model,roc_auc,pr_auc,accuracy
0,Logistic regression,0.5445,0.5497,0.5376
1,LightGBM,0.6252,0.6349,0.5877


解讀

Logistic regression：ROC-AUC 0.545，只比亂猜（0.5）好一點。Accuracy 53.8%，比「全部猜會買」的 51.98% 只高 1.8 個百分點
LightGBM：ROC-AUC 0.625，accuracy 58.8%，比 baseline 高約 6.8 個百分點

LightGBM 明顯比較好，但要誠實看待：0.625 是中等偏弱的預測力。目前的 features 只有 session 內 cart 前的行為和商品屬性，資訊量有限。

兩個 model 的差距可能來自兩個原因，現在還分不清楚：

LightGBM 能抓到非線性關係（例如價格對購買率的影響不是直線），logistic regression 不行。而且 secs_to_first_cart 這種長尾分布的 feature，沒取 log 之前對 logistic regression 很不友善
LightGBM 用了 cart_category_id，logistic regression 沒有

### Feature importance

In [5]:
# gain：每個 feature 在所有分裂中帶來的總改善量，越大越重要
importance = pd.DataFrame({
    'feature': X_train.columns,
    'gain': model.booster_.feature_importance(importance_type='gain')
}).sort_values('gain', ascending=False)

importance['gain_pct'] = (100 * importance['gain'] / importance['gain'].sum()).round(1)
importance

,feature,gain,gain_pct
2,secs_to_first_cart,120996.914778,37.1
7,cart_category_id,120501.652584,36.9
5,cart_price,32372.516845,9.9
3,cart_hour,24907.047224,7.6
0,n_views_before_cart,11180.617932,3.4
4,cart_dow,8143.598657,2.5
1,n_products_viewed_before_cart,7974.140279,2.4
6,cart_brand_missing,73.942161,0.0


### 拿掉類別再跑一次 LightGBM

In [6]:
# 只用和 logistic regression 相同的 features，讓兩個 model 公平比較
model_nocat = lgb.LGBMClassifier(n_estimators=300, learning_rate=0.05, num_leaves=31, verbose=-1)
model_nocat.fit(train[num_features], y_train)

p_nocat = model_nocat.predict_proba(test[num_features])[:, 1]
results.append(evaluate('LightGBM (no category)', y_test, p_nocat))
pd.DataFrame(results)

,model,roc_auc,pr_auc,accuracy
0,Logistic regression,0.5445,0.5497,0.5376
1,LightGBM,0.6252,0.6349,0.5877
2,LightGBM (no category),0.6172,0.6287,0.5812


secs_to_first_cart 是最重要的 feature，從進站到加入購物車花多久，跟最後會不會買有關。這個關係很可能是非線性的（太快和太慢都可能代表不同的意圖），之後可以做一個 EDA 看各時間區間的購買率，會是 pitch 裡很具體的發現
cart_brand_missing 幾乎是 0，品牌缺值對購買沒有預測力，可以在 README 寫一句驗證過了

### 合併使用者歷史 features，重新訓練

In [7]:
# 讀取使用者歷史 features，依 session 接到原本的表
hist = pd.read_parquet('../data/processed/feat_user_history.parquet')
df2 = df.merge(hist, on='user_session', how='left')

train2 = df2[df2['split'] == 'train']
test2 = df2[df2['split'] == 'test']

# 原本的數值 features 加上三個歷史 features
num_features_v2 = num_features + ['prior_purchases', 'prior_carts', 'prior_sessions']

# Logistic regression
lr2 = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
lr2.fit(train2[num_features_v2], y_train)
results.append(evaluate('Logistic regression + history', y_test, lr2.predict_proba(test2[num_features_v2])[:, 1]))

# LightGBM（含類別）
def prep_v2(d):
    X = d[num_features_v2].copy()
    X['cart_category_id'] = pd.Categorical(d['cart_category_id'], categories=cats)
    return X

X_train2, X_test2 = prep_v2(train2), prep_v2(test2)

model2 = lgb.LGBMClassifier(n_estimators=300, learning_rate=0.05, num_leaves=31, verbose=-1)
model2.fit(X_train2, y_train)
p_lgb2 = model2.predict_proba(X_test2)[:, 1]
results.append(evaluate('LightGBM + history', y_test, p_lgb2))

pd.DataFrame(results)

/var/folders/dc/03c0xx517lscty57znvb7_8m0000gn/T/ipykernel_30314/3025603374.py:19: Pandas4Warning: Constructing a Categorical with a dtype and values containing non-null entries not in that dtype's categories is deprecated and will raise in a future version.
  X['cart_category_id'] = pd.Categorical(d['cart_category_id'], categories=cats)


,model,roc_auc,pr_auc,accuracy
0,Logistic regression,0.5445,0.5497,0.5376
1,LightGBM,0.6252,0.6349,0.5877
2,LightGBM (no category),0.6172,0.6287,0.5812
3,Logistic regression + history,0.6955,0.7250,0.6396
4,LightGBM + history,0.7299,0.7507,0.6676


解讀

使用者是誰，比這次 session 做了什麼更能預測結帳。這一批只有 3 個 feature，帶來的提升比前面 8 個 feature 加起來還多。這是 project 最核心的發現，pitch 和 README 都應該以這點為主軸
Logistic regression 進步得更多，兩個 model 的差距從 0.08 縮小到 0.03。代表歷史 features 和購買之間大致是線性關係（買過越多次的人越可能結帳），簡單的 model 也抓得到
要寫進 README 的一個注意事項：因為資料從 10/1 才開始，月底的 test set 使用者平均累積了較多歷史，月初的 train set 則大多是 0。兩邊的 feature 分布不完全一致，有可能讓 test 的結果略為樂觀。加入 9 月資料可以驗證這點

### Model 抓出放棄者的能力

np.argsort(-p_abandon)：依放棄機率由高到低排序後的索引。前面加負號，是因為 argsort 預設由小到大
abandon_rate：model 認為最可能放棄的前 k% 裡，實際放棄的比例
lift：前 k% 的放棄率是整體放棄率的幾倍。top_pct = 100 那列就是全部 session，lift 一定是 1.0

In [8]:
# 預測「放棄」的機率 = 1 - 預測購買的機率
p_abandon = 1 - p_lgb2
abandoned = 1 - y_test.values
base_rate = abandoned.mean()          # 整體放棄率

# 依放棄機率由高到低排序，看前 k% 的 session 實際放棄率是多少
order = np.argsort(-p_abandon)
rows = []
for k in [0.1, 0.2, 0.3, 0.5, 1.0]:
    n = int(len(order) * k)
    rate = abandoned[order[:n]].mean()
    rows.append({
        'top_pct': int(k * 100),
        'n_sessions': n,
        'abandon_rate': round(rate, 4),
        'lift': round(rate / base_rate, 2),
    })

pd.DataFrame(rows)

,top_pct,n_sessions,abandon_rate,lift
0,10,10987,0.7611,1.58
1,20,21975,0.7236,1.51
2,30,32962,0.6973,1.45
3,50,54937,0.6474,1.35
4,100,109875,0.4802,1.00
